# 🔴 Interview: Ring Attention

Sequence-parallel attention via ring communication, simulated with online softmax

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def ring_attention(Q, K, V, num_devices):
    """
    手写环形注意力 —— 面试版。

    面试考点:
    1. 序列并行：把长序列分到多个设备上，减少单设备内存
    2. Online Softmax 是核心难点：如何在不存储完整注意力矩阵的情况下累加结果
    3. 环形通信：K/V 在设备间轮转，每个设备看到所有 K/V
    4. m_new 的更新和 l/o 的校正因子

    参数: Q, K, V: (B, S, D), num_devices: int
    返回: (B, S, D)
    """
    B, S, D = Q.shape
    chunk = S // num_devices
    scale = D ** -0.5

    Q_chunks = Q.split(chunk, dim=1)
    K_chunks = K.split(chunk, dim=1)
    V_chunks = V.split(chunk, dim=1)

    outputs = []
    for i, Qi in enumerate(Q_chunks):
        # Online softmax 三件套: 最大值 m, 归一化因子 l, 输出 o
        m = torch.full((B, chunk, 1), float('-inf'), device=Q.device)
        l = torch.zeros(B, chunk, 1, device=Q.device)
        o = torch.zeros(B, chunk, D, device=Q.device)

        for j in range(num_devices):
            Kj = K_chunks[(i + j) % num_devices]
            Vj = V_chunks[(i + j) % num_devices]

            # 局部注意力分数
            scores = (Qi @ Kj.transpose(-2, -1)) * scale  # (B, chunk, chunk)

            # Online softmax 更新
            m_new = torch.maximum(m, scores.max(dim=-1, keepdim=True).values)
            exp_s = torch.exp(scores - m_new)
            correction = torch.exp(m - m_new)  # 旧最大值到新最大值的校正因子
            l_new = l * correction + exp_s.sum(dim=-1, keepdim=True)
            o = o * (l * correction) / l_new + (exp_s @ Vj) / l_new
            m, l = m_new, l_new

        outputs.append(o)

    return torch.cat(outputs, dim=1)

In [ ]:
# Verify numerical equivalence with standard attention for 2 and 4 devices
torch.manual_seed(42)
B, S, D = 2, 8, 16
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

# Standard attention reference
scale = D ** -0.5
scores_ref = (Q @ K.transpose(-2, -1)) * scale
ref_out = torch.softmax(scores_ref, dim=-1) @ V

for num_devices in [2, 4]:
    ring_out = ring_attention(Q, K, V, num_devices=num_devices)
    max_diff = (ring_out - ref_out).abs().max().item()
    match = torch.allclose(ring_out, ref_out, atol=1e-5)
    print(f'num_devices={num_devices}  shape={tuple(ring_out.shape)}  max_diff={max_diff:.2e}  match={match}')

In [ ]:
from torch_judge import check
check('ring_attention')

## 📝 核心思路总结

1. **序列并行**：把长序列分块到多个设备上，每块只存自己的 Q，K/V 在环上轮转，每个设备都能「看到」所有 K/V。
2. **Online Softmax 是核心**：不需要一次性计算完整 S×S 注意力矩阵，而是逐块累加，用 `m_new = max(m, scores.max)` 更新最大值并校正之前的累积值。
3. **校正因子 `exp(m - m_new)`**：当最大值更新时，之前累积的 l 和 o 都要乘以这个因子来补偿最大值的变化。
4. **环形通信模式**：K/V 块按环形顺序传递，每个 Q 块恰好遍历所有 K/V 块一次，总通信量与设备数成正比。